# Deposit Attrition EDA — v7

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v6 answered the *scientific* question and left the *operational* one untouched. Everything in
v6 §4 and §5 was measured in **event time** — statistics inside a `rel_m` cell, against stayers
handed a pseudo-event. In production nobody knows `rel_m`. You score the whole live book every
calendar month and find out later who was right.

v7 moves the spine of the analysis to **calendar time** and builds the four open items on top
of it.

## What that change exposes immediately

`fin_out_n` — the champion signal, 30.1× lift — has **34.7% coverage**. In event time that is a
footnote in the ordering table. In a queue it means **roughly two thirds of the at-risk book
cannot be scored by it at all**. `bal_live` covers 73.6%. A queue built on the best single
feature is a queue that never looks at most of the book, and no v6 table says so, because
coverage was only ever used to gate an ordering claim.

That reframes §4.2. The case for a fitted model is not that it out-lifts `fin_out_n` where both
are defined. It is that missingness is itself a feature — a customer with no `fin_out_n` dd does
not pay institutions in a way we can measure — and a model that carries missingness indicators
scores **100% of the book**. §4 tests whether that is worth anything.

## The blocks

| § | Open item | What v7 does |
|---|---|---|
| 2 | §4.1 — four signals separate at exactly −12 | Widen to `rel_m −18` on the subset with enough history, **with a within-cohort control**: the same restricted cohort re-read at the v6 window. Cohort drift is the trap that killed the v1 locatability k-curve; it is not repeated here |
| 3 | — | Build the calendar-time risk set: one row per (customer, month) while live and pre-event, wide dd, missingness flags, forward labels at H ∈ {1,3,6} |
| 4 | §4.2 — "combining doesn't help" is only proven for an unweighted count | Discrete-time hazard, **rolling origin**, five specs: `bal_live` alone · `fin_out_n` alone · v6's unweighted count · payments-only · payments + balance. Train on `t ≤ T−H`, test on `t = T`. Nothing at or after the origin enters training |
| 5 | §4.3 — nobody priced the queue | Capacity-constrained: precision / recall / alerts-per-TP / lead at K alerts a month, two-tier design, overlap between tiers, **alert fatigue** (repeat-flagging), and the incumbent 30% rule measured in the same calendar frame |
| 6 | §5.4 — validate on `B_bal_exit` | Same risk set carries both event columns; labels re-derived, no rebuild |
| 7 | §4.4 — competitor vs contraction | **Candidate segmentation only.** Counterparty survival from `pay_pairs` × ticket-hold, against a stayer baseline. There is no ground truth, and the section says so in its own output |

Also fixed: the §4.5 `kv()` duplicate-key bug — `kv` now takes pairs and **raises** on a repeated
label rather than silently printing the last one.

## What v7 does not do

It does not rebuild panels, payment features or labels. It reads v2's panels, v3/v5/v6's labels
and v6's frozen peer anchor, and rebuilds only what is missing. It does not touch counterparty
data — `PAYS_CPTY` and `CptyFinEntity` remain the thing that would let signal 12 be measured
properly rather than from the PNC-visible slice.


## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v7
# =====================================================================
# v6 measured operating points inside rel_m cells. That answers "how
# different is an attriter 6 months out", which is a scientific question,
# and it does NOT answer "what does the alert list look like in March".
# Three things go wrong in the translation and all three are silent:
#
#   1. The stayer side is a matched pseudo-event sample, not the book.
#      Precision computed against it is a statement about a designed
#      comparison, not about the population you would actually score.
#   2. A feature's coverage becomes a gate on the ORDERING claim but
#      never a gate on the QUEUE. fin_out_n covers 34.7% of rows -
#      in production that is two thirds of the book with no score.
#   3. Nothing in event time prices repeat-flagging. A customer flagged
#      in six consecutive months is one relationship conversation, not
#      six, and the alerts-per-true-positive number depends on which
#      you mean.
#
# v7 scores every at-risk customer-month, ranks WITHIN the month, and
# reports at a fixed monthly capacity. That is the object TM Sales
# receives.
from pathlib import Path

HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_V3  = "hdfs://nameservice1/user/pk36814/attrition_v3"
HDFS_V5  = "hdfs://nameservice1/user/pk36814/attrition_v5"
HDFS_V6  = "hdfs://nameservice1/user/pk36814/attrition_v6"
HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v7"
# TRAP (02 §6): Path() collapses hdfs://host/p -> hdfs:/host/p and the
# job dies with Permission denied inode="/". Local Path and HDFS string
# are separate config vars and never mixed.
OUT_DIR  = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v7")

DATE_START, DATE_END = "2024-01-01", "2026-07-31"
MAX_ROWS, ZERO_TOL   = 60, 1.0
SEED = 20260906

# ── carried unchanged from v6 so dd is byte-comparable ────────────────
CHG_LAG_FAR, CHG_LAG_NEAR = -6, -4
CHG_MIN_OBS, MIN_REF      = 2, 1.0
PEER_DECILES, PEER_ANCHOR_M, PEER_MIN_N = 10, 3, 50
PEER_USE_SEG = False          # segment_desc is a leakage risk: restatement
                              # lift 21.7 at rel_m -1. It is not a
                              # point-in-time attribute and never enters.
MIN_HIST_M, MIN_LIVE_BEFORE  = 12, 6
BAL_EXIT_FRAC, BAL_EXIT_HOLD = 0.05, 3
P30_DROP     = 0.70
BURN_IN_YM   = ["2024-01", "2024-02", "2024-03"]
NEW_ENTITY   = ["cpty_new_out", "fin_new_out"]
NO_RATIO     = ["net_flow"]
MIN_CELL_N, MIN_COVERAGE  = 200, 0.25
SEP_LEVEL, SEP_RATE, HOLD = 0.15, 0.05, 2
SCORE_THRESH = 0.70           # v6's unweighted count rule, carried as a baseline

# ── §2 · widened pre-window ───────────────────────────────────────────
EVENT_PRE, EVENT_POST = 12, 3     # v6's window, kept as the control
WIDE_PRE   = 18                   # 31-month panel, 12-month label burn-in
SEARCH_FROM_WIDE = -WIDE_PRE
DD_FIRST_M = 6                    # dd needs t-6..t-4, so no dd below this

# ── §3 · calendar-time risk set ───────────────────────────────────────
HORIZONS   = [1, 3, 6]            # event within (t, t+H]
PRIMARY_H  = 6
EVAL_DEFS  = ["A_full_exit", "B_bal_exit"]
PRIMARY_DEF = "A_full_exit"
# Row-level negative sampling for TRAINING only. Evaluation runs on the
# full book, so precision is exact and needs no weighting. The intercept
# is prior-corrected by ln(NEG_SAMPLE) - King & Zeng.
NEG_SAMPLE = 0.15
# Escape hatch if the per-month collect is slow on the cluster. 1.0 =
# the whole book. Anything below that makes precision an estimate and
# the notebook says so in the output.
TEST_STAYER_FRAC = 1.0
MAX_COLLECT_ROWS = 3_000_000

# ── §4 · rolling origin ───────────────────────────────────────────────
ORIGIN_START   = 18       # first test month; train is t <= T-H
MIN_TRAIN_POS  = 300      # a fold with fewer positives is not scored
L2             = 2.0      # ridge on standardised features, intercept free
IRLS_MAX_IT, IRLS_TOL = 60, 1e-9
DD_CLIP        = (0.01, 100.0)

# ── §5 · the queue ────────────────────────────────────────────────────
CAPACITY  = [50, 100, 250, 500, 1000, 2500]
QUEUE_K   = 250           # the headline capacity
TIER1_K, TIER2_K = 250, 1000
COOLDOWN_M = 3            # a customer re-alerted within this many months
                          # is the SAME alert, not a new one

# ── §7 · competitor vs contraction ────────────────────────────────────
RUN_ATTRIB  = True
PP_KT_CPTY  = "cpty"      # CORRECT THESE FROM THE 7a PROBE BEFORE TRUSTING 7c
PP_KT_FIN   = "fin"
PP_FLOW_OUT = "out"
ATTRIB_BASE = (-12, -7)   # baseline counterparty set
ATTRIB_POST = (0, 3)      # survival window
ATTRIB_MAX_CPTY = 50      # cap per customer; long tails dominate the join

# ── the twelve, unchanged ─────────────────────────────────────────────
SIGNALS = [
 dict(n=1,  name="Stops taking on new trading partners",     feature="cpty_new_out",    rule="zero"),
 dict(n=2,  name="Stops paying anyone at an unfamiliar bank", feature="fin_new_out",     rule="zero"),
 dict(n=3,  name="First rail to go — cheque",                 feature="amt_out_check",   rule="dd"),
 dict(n=4,  name="Net flow turns against us",                 feature="net_flow",        rule="sign"),
 dict(n=5,  name="Their own customers stop paying them here", feature="amt_in_internal", rule="dd"),
 dict(n=6,  name="Spending through us falls",                 feature="amt_out",         rule="dd"),
 dict(n=7,  name="Fewer payments, not just smaller",          feature="n_out",           rule="dd",
                                                              companion="avg_ticket_out"),
 dict(n=8,  name="Inbound activity thins",                    feature="n_in",            rule="dd"),
 dict(n=9,  name="What today's monitoring sees",              feature="bal_live",        rule="dd"),
 dict(n=10, name="Payments to other PNC customers fall",      feature="amt_out_internal",rule="dd"),
 dict(n=11, name="The relationship list itself shrinks",      feature="cpty_out_n",      rule="dd"),
 dict(n=12, name="Banks they pay drop away",                  feature="fin_out_n",       rule="dd"),
]
RAILS      = ["ach", "wire", "check", "card", "rtp", "other"]
RAIL_FEATS = [f"amt_out_{r}" for r in RAILS]
SIG_FEATS  = [s["feature"] for s in SIGNALS] + ["avg_ticket_out", "avg_ticket_in", "amt_in"]
# The four whose onset v6 could not see. §2 exists for these.
WALL_FEATS = ["amt_out_check", "amt_in_internal", "cpty_new_out", "fin_new_out"]
HTML_NAME  = "PKG_Attrition_Queue.html"


In [ ]:
# =====================================================================
# 1 · IMPORTS, HELPERS, NUMPY-2 / SPARK-3.3 COMPAT
# =====================================================================
import warnings, html as _html, time, math, json, datetime as dt
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)
spark = (SparkSession.builder.appName("pkg_attrition_eda_v7")
         .config("spark.sql.shuffle.partitions", "400")
         # KEEP THIS OFF. PySpark 3.3.2's arrow conversion path still
         # references np.object0 / np.bool8, both removed in numpy 2.0.
         # It is not only the decimal(15,0) case - the whole arrow path
         # is unsafe on this pairing. Everything here collects through
         # the row path and casts decimals first (_dec).
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .enableHiveSupport().getOrCreate())
pd.set_option("display.max_columns", 300); pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"numpy {np.__version__} · pandas {pd.__version__} · spark {spark.version}")
assert int(np.__version__.split(".")[0]) >= 2, "compat notes below assume numpy 2.x"
# numpy 2 removals this notebook deliberately routes around:
#   np.trapz    -> AUC is computed from ranks (exact, and no trapezoid)
#   np.float_ / np.NaN / np.Inf / np.object0 / np.bool8 -> never referenced
#   np.in1d     -> np.isin
#   np.array(x, copy=False) now RAISES -> np.asarray everywhere

def hp(n): return f"{HDFS_DIR.rstrip('/')}/{n}"
def v2(n): return f"{HDFS_V2.rstrip('/')}/{n}"
def v3(n): return f"{HDFS_V3.rstrip('/')}/{n}"
def v5(n): return f"{HDFS_V5.rstrip('/')}/{n}"
def v6(n): return f"{HDFS_V6.rstrip('/')}/{n}"
def pct(a, b): return float(a)/float(b) if b else float("nan")
def sdiv(a, b): return F.when(F.col(b) > 0, F.col(a)/F.col(b))

def _dec(s):
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o

def read_first(paths, label):
    """First path that exists wins. Returns None if none do, so the caller
    decides whether to rebuild or fail - never a silent empty frame."""
    for p in paths:
        try:
            d = spark.read.parquet(p); d.limit(1).count()
            print(f"  {label}: reusing {p}"); return d
        except Exception:
            continue
    print(f"  {label}: not found in {len(paths)} location(s) — rebuilding")
    return None

def collect_pd(sdf, label="", max_rows=None):
    """Full collect with the row count printed BEFORE the wait, and a hard
    ceiling. The arrow path is off, so this is the row path: budget
    roughly a minute per 200-300k rows at ~50 numeric columns."""
    max_rows = MAX_COLLECT_ROWS if max_rows is None else max_rows
    n = sdf.count()
    if n > max_rows:
        raise RuntimeError(f"{label}: {n:,} rows exceeds MAX_COLLECT_ROWS={max_rows:,}. "
                           f"Lower TEST_STAYER_FRAC or narrow the month range.")
    t0 = time.time(); out = _dec(sdf).toPandas()
    print(f"  collected {label}: {n:,} rows x {out.shape[1]} cols in {time.time()-t0:,.0f}s")
    return out

def disp(obj, title=None, n=None, save=None, transpose=False):
    n = MAX_ROWS if n is None else n
    out = _dec(obj).limit(n).toPandas() if hasattr(obj, "toPandas") else (
        obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;margin:10px 0 2px;"
                     f"color:#111'>{title}<span style='font-weight:400;color:#888'> &middot; "
                     f"{len(out)} rows</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(pairs, title=None, save=None):
    """FIX (01 §4.5). v6 passed a dict literal with two identical
    "  ...of" keys, so the first two counts were silently dropped and the
    table printed 3 where it should have printed 12 and 7. This takes an
    ORDERED LIST OF PAIRS and raises on a repeated label - the failure is
    now loud and at authoring time. A dict is still accepted; a dict
    literal simply cannot carry the bug past the parser."""
    items = list(pairs.items()) if isinstance(pairs, dict) else list(pairs)
    labs = [k for k, _ in items]
    dupes = sorted({k for k in labs if labs.count(k) > 1})
    if dupes: raise ValueError(f"kv(): duplicate labels {dupes} — give each row a distinct label")
    return disp(pd.DataFrame({"metric": labs, "value": [v for _, v in items]}),
                title=title, n=len(items), save=save)

# ── metrics ───────────────────────────────────────────────────────────
def auc(y, s):
    """Mann-Whitney AUC from ranks. Exact, ties handled, and no np.trapz
    (removed in numpy 2). Rows with a non-finite score are DROPPED, not
    imputed - an unscoreable customer is not a coin flip, and the share
    dropped is reported separately as queue coverage."""
    y = np.asarray(y, dtype=float); s = np.asarray(s, dtype=float)
    ok = np.isfinite(s) & np.isfinite(y)
    y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y) - n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1*(n1+1)/2.0) / (n1*n0))

def topk_metrics(score, y, ks, p_base=None, unscoreable_last=True):
    """Rank WITHIN the frame handed in - which is one calendar month.
    Unscoreable rows sort last and still count in the denominator of
    coverage, because in production they are customers you were unable
    to look at, not customers you chose not to alert."""
    s = np.asarray(score, dtype=float); y = np.asarray(y, dtype=float)
    n_all = len(s); fin = np.isfinite(s)
    if unscoreable_last:
        s = np.where(fin, s, -np.inf)
    order = np.argsort(-s, kind="stable")
    ys, ss = y[order], s[order]
    tot = float(y.sum()); p_base = (tot/n_all) if p_base is None else p_base
    rows = []
    for K in ks:
        k = int(min(K, n_all))
        tp = float(ys[:k].sum()); scored = int(np.isfinite(ss[:k]).sum())
        prec = tp/k if k else np.nan
        rows.append(dict(k=K, n_at_risk=n_all, n_scoreable=int(fin.sum()),
                         queue_coverage=fin.mean(), n_alerts_scoreable=scored,
                         tp=int(tp), precision=prec,
                         recall=(tp/tot if tot else np.nan),
                         lift=(prec/p_base if p_base > 0 else np.nan),
                         alerts_per_tp=(k/tp if tp > 0 else np.nan)))
    return pd.DataFrame(rows)

# ── the model ─────────────────────────────────────────────────────────
def logit_irls(X, y, l2=L2, max_iter=IRLS_MAX_IT, tol=IRLS_TOL):
    """Newton-Raphson with a ridge penalty. X carries its own intercept
    column at position 0 and that column is NOT penalised. Pure numpy on
    purpose - no sklearn dependency on the cluster, and a 50x50 normal
    equation is free at any row count we reach here."""
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y, dtype=np.float64)
    n, p = X.shape
    b = np.zeros(p); R = l2*np.eye(p); R[0, 0] = 0.0
    for _ in range(max_iter):
        eta = np.clip(X @ b, -30.0, 30.0)
        mu  = 1.0/(1.0 + np.exp(-eta))
        w   = np.maximum(mu*(1.0-mu), 1e-6)
        z   = eta + (y - mu)/w
        XtW = X.T * w
        try:
            bn = np.linalg.solve(XtW @ X + R, XtW @ z)
        except np.linalg.LinAlgError:
            bn = np.linalg.lstsq(XtW @ X + R, XtW @ z, rcond=None)[0]
        if np.max(np.abs(bn - b)) < tol: b = bn; break
        b = bn
    return b

def prior_correct(b0, s):
    """King & Zeng. Positives are kept whole and negatives sampled at s,
    so sample odds are population odds / s and the intercept shifts by
    ln(s). Ranking is unaffected; PRECISION is not, which is the whole
    point of correcting it."""
    return b0 + math.log(s)

def fit_spec(tr, cols, s_neg):
    """Standardise on train, fit, un-standardise so the returned beta
    applies to raw columns. Zero-variance columns are dropped here rather
    than earlier: net_flow's dd is null by design (NO_RATIO) and
    amt_out_other is empty, so both arrive constant and would make the
    normal equation singular."""
    X = tr[cols].to_numpy(dtype=np.float64)
    y = tr["y"].to_numpy(dtype=np.float64)
    sd = X.std(axis=0); keep = sd > 1e-9
    cols_k = [c for c, k in zip(cols, keep) if k]
    if not cols_k: return None
    Xk = X[:, keep]; mu = Xk.mean(axis=0); sdk = Xk.std(axis=0)
    Z  = np.column_stack([np.ones(len(Xk)), (Xk - mu)/sdk])
    b  = logit_irls(Z, y)
    beta = b[1:]/sdk
    b0   = prior_correct(float(b[0] - float(np.sum(b[1:]*mu/sdk))), s_neg)
    return dict(cols=cols_k, beta=beta, b0=b0, n=len(tr), n_pos=int(y.sum()))

def apply_spec(spec, df):
    if spec is None: return np.full(len(df), np.nan)
    X = df[spec["cols"]].to_numpy(dtype=np.float64)
    return spec["b0"] + X @ spec["beta"]

# ── findings ledger ───────────────────────────────────────────────────
_F = OUT_DIR / "FINDINGS_v7.csv"
FINDINGS = pd.read_csv(_F).to_dict("records") if _F.exists() else []
def note(qid, q, a, d=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=q, answer=str(a), detail=str(d)))
    pd.DataFrame(FINDINGS).to_csv(_F, index=False)

def tbl(df, cls="t"):
    return df.to_html(index=False, classes=cls, border=0,
                      float_format=lambda v: f"{v:,.3f}", na_rep="&mdash;", escape=False)
print("helpers ready")


## 1 · dd, labels, and the incumbent as a monthly flag

In [ ]:
# =====================================================================
# 2 · dd + LABELS + THE INCUMBENT, MONTHLY               [OUTPUT BLOCK 1]
# =====================================================================
# Reuses v2's panels, v6's frozen peer anchor and v5/v6's labels. The one
# new object is the incumbent rule as a PER-MONTH FLAG. v6 measured the
# 30% rule as a lead time (median 2 months) which is an event-time
# statistic; §5 needs to know how many customers it fires on in March,
# and with what precision, which is a calendar-time statistic. They are
# not the same question and only the second one sizes a queue.
t0 = time.time()
cust_month = spark.read.parquet(v2("panel_customer_month")).filter(F.col("ym") >= DATE_START[:7])
feat       = spark.read.parquet(v2("panel_pay_features")).filter(F.col("ym") >= DATE_START[:7])
for c in NEW_ENTITY:
    feat = feat.withColumn(c, F.when(F.col("ym").isin(*BURN_IN_YM), None).otherwise(F.col(c)))

panel = (cust_month.select("cust_pwr_id", "ym", "m_idx", "bal_live", "n_accts",
                           "n_accts_live", "all_closed", "segment_desc", "naics", "state")
         .join(feat.drop("ym"), ["cust_pwr_id", "m_idx"], "left")
         .withColumn("avg_ticket_out", sdiv("amt_out", "n_out"))
         .withColumn("avg_ticket_in",  sdiv("amt_in",  "n_in"))).persist(StorageLevel.DISK_ONLY)

FEATS  = sorted(set(SIG_FEATS + RAIL_FEATS) & set(panel.columns) | {"bal_live"})
_stack = ", ".join([f"'{f}', CAST({f} AS DOUBLE)" for f in FEATS])
M_MIN, M_MAX = [int(x) for x in panel.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]

# ── frozen peer anchor ────────────────────────────────────────────────
anchor = read_first([v6("peer_anchor"), hp("peer_anchor")], "peer_anchor")
if anchor is None:
    wc = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
    anchor = (panel.withColumn("rn", F.row_number().over(wc)).filter(F.col("rn") <= PEER_ANCHOR_M)
              .groupBy("cust_pwr_id").agg(F.avg("bal_live").alias("anchor_bal"),
                                          F.max("segment_desc").alias("anchor_seg")))
    anchor = (anchor.withColumn("bal_decile", F.ntile(PEER_DECILES).over(
                       Window.orderBy(F.col("anchor_bal").asc_nulls_first())))
              .withColumn("peer_key", F.concat_ws("|", F.col("bal_decile").cast("string"),
                          F.coalesce("anchor_seg", F.lit("NA")) if PEER_USE_SEG else F.lit("ALL")))
              .select("cust_pwr_id", "peer_key", "bal_decile", "anchor_bal"))
    anchor.write.mode("overwrite").parquet(hp("peer_anchor"))
    anchor = spark.read.parquet(hp("peer_anchor"))
anchor = anchor.persist(StorageLevel.DISK_ONLY)

# ── difference-in-differences, identical construction to v6 ───────────
long = (panel.join(anchor, "cust_pwr_id", "left")
        .select("cust_pwr_id", "m_idx", "ym", "peer_key",
                F.expr(f"stack({len(FEATS)}, {_stack}) as (feature, value)")))
wr = (Window.partitionBy("cust_pwr_id", "feature").orderBy("m_idx")
      .rangeBetween(CHG_LAG_FAR, CHG_LAG_NEAR))   # rangeBetween, never rowsBetween:
                                                  # rows shift where a month is missing
long = (long.withColumn("ref",  F.avg("value").over(wr))
             .withColumn("nref", F.count("value").over(wr))
             .withColumn("chg", F.when((F.col("nref") >= CHG_MIN_OBS) &
                                       (F.col("ref") > MIN_REF) &
                                       (~F.col("feature").isin(*NO_RATIO)),
                                       F.col("value")/F.col("ref"))))
peer_chg = (long.groupBy("ym", "peer_key", "feature")
            .agg(F.count("chg").alias("peer_n"),
                 F.expr("percentile_approx(chg, 0.5)").alias("peer_chg"))
            .filter((F.col("peer_n") >= PEER_MIN_N) & (F.abs(F.col("peer_chg")) > 1e-6)))
long = (long.join(peer_chg, ["ym", "peer_key", "feature"], "left")
        .withColumn("dd", F.when(F.col("chg").isNotNull() & F.col("peer_chg").isNotNull(),
                                 F.col("chg")/F.col("peer_chg")))
        ).persist(StorageLevel.DISK_ONLY)

cov = (long.groupBy("feature").agg(
          F.count("*").alias("rows"),
          F.avg(F.col("dd").isNotNull().cast("double")).alias("share_with_dd"),
          F.expr("percentile_approx(dd, 0.5)").alias("median_dd")).orderBy("feature"))
COV = disp(cov, title="1a &middot; dd coverage — median_dd at 1.000 is the construction check. "
                      "<b>share_with_dd is also the ceiling on queue coverage for that feature</b>, "
                      "which is what §5 is about", n=40, save="v7_dd_coverage")

# ── labels ────────────────────────────────────────────────────────────
w    = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
w3   = w.rangeBetween(-2, 0); w6p = w.rangeBetween(-8, -3); w12 = w.rangeBetween(-11, 0)
wfwd = w.rangeBetween(0, BAL_EXIT_HOLD - 1)
c = (cust_month
     .withColumn("avg3", F.avg("bal_live").over(w3)).withColumn("n3", F.count("bal_live").over(w3))
     .withColumn("prior6", F.avg("bal_live").over(w6p)).withColumn("n_prior6", F.count("bal_live").over(w6p))
     .withColumn("n_hist", F.count("bal_live").over(w12))
     .withColumn("_a", F.array_sort(F.collect_list("bal_live").over(w12)))
     .withColumn("med12", F.expr("element_at(_a, cast(size(_a)/2 as int) + 1)")).drop("_a")
     .withColumn("low", ((F.col("med12") > ZERO_TOL) &
                         (F.col("bal_live") < BAL_EXIT_FRAC*F.col("med12"))).cast("int"))
     .withColumn("low_run", F.sum("low").over(wfwd)).withColumn("obs_fwd", F.count("*").over(wfwd))
     .withColumn("acr", F.sum("all_closed").over(wfwd)))
DEFS = {
 "A_full_exit": (F.col("all_closed") == 1) & (F.col("acr") == F.col("obs_fwd")),
 "B_bal_exit":  (F.col("n_hist") >= MIN_HIST_M) & (F.col("low_run") == BAL_EXIT_HOLD) &
                (F.col("obs_fwd") == BAL_EXIT_HOLD),
 "C_p30":       (F.col("n_hist") >= MIN_HIST_M) & (F.col("n_prior6") >= 4) & (F.col("n3") >= 3) &
                (F.col("avg3") < P30_DROP*F.col("prior6")),
}
for k, cond in DEFS.items(): c = c.withColumn(k, F.when(cond, 1).otherwise(0))

# NEW: the incumbent as a monthly flag, persisted. v6 collapsed C_p30 to
# a first-fire month and lost the fact that it re-fires. A rule that
# fires on the same customer eleven months running is one alert or
# eleven depending on how you count, and §5 prices both.
p30_month = c.select("cust_pwr_id", "m_idx", F.col("C_p30").alias("p30_fire"))
p30_month.write.mode("overwrite").parquet(hp("p30_month"))
p30_month = spark.read.parquet(hp("p30_month")).persist(StorageLevel.DISK_ONLY)

lab = read_first([v6("labels_customer"), v5("labels_customer")], "labels_customer")
if lab is None:
    lab = (c.groupBy("cust_pwr_id").agg(
              *[F.min(F.when(F.col(k) == 1, F.col("m_idx"))).alias(f"m_{k}") for k in DEFS],
              F.min("m_idx").alias("first_m"), F.max("m_idx").alias("last_m"),
              F.count("*").alias("n_months"),
              F.min(F.when(F.col("n_accts_live") > 0, F.col("m_idx"))).alias("first_live_m"),
              F.max("segment_desc").alias("segment_desc"))
           .filter(f"n_months >= {MIN_HIST_M}"))
    for k in DEFS:
        lab = lab.withColumn(f"q_{k}", F.when(
            F.col(f"m_{k}").isNotNull() & F.col("first_live_m").isNotNull() &
            (F.col(f"m_{k}") - F.col("first_live_m") >= MIN_LIVE_BEFORE), F.col(f"m_{k}")))
    lab.write.mode("overwrite").parquet(hp("labels_customer"))
    lab = spark.read.parquet(hp("labels_customer"))
lab = lab.persist(StorageLevel.DISK_ONLY)

N_EV   = lab.count()
EVAL_M = M_MAX - M_MIN + 1 - MIN_HIST_M
PREV   = {d: pct(lab.filter(F.col(f"q_{d}").isNotNull()).count(), N_EV*EVAL_M) for d in EVAL_DEFS}
disp(pd.DataFrame([dict(definition=k,
                        n_raw=lab.filter(F.col(f"m_{k}").isNotNull()).count(),
                        n_qualified=lab.filter(F.col(f"q_{k}").isNotNull()).count(),
                        monthly_hazard=PREV.get(k)) for k in DEFS]),
     title=f"1b &middot; Labels (n={N_EV:,} customers, {EVAL_M} evaluable months, "
           f"m_idx {M_MIN}&ndash;{M_MAX})", save="v7_labels")

kv([("customer-months", panel.count()),
    ("features carried", len(FEATS)),
    ("peer grouping", f"{PEER_DECILES} FROZEN deciles, segment EXCLUDED (leakage)"),
    ("dd first available at m_idx", DD_FIRST_M),
    ("block 1 wall (s)", round(time.time()-t0))],
   title="1c &middot; Construction", save="v7_construction")
note("V7SPINE", "What changes in v7?",
     "the unit of analysis moves from event time to calendar time",
     "Event-time operating points measure a designed comparison against pseudo-event stayers. "
     "A queue is a rank within a calendar month over the whole at-risk book. §5 is the first "
     "table in this programme that a sales manager could act on directly.")


## 2 · Where does the −12 wall actually start?

`amt_out_check`, `amt_in_internal`, `cpty_new_out` and `fin_new_out` all separate at exactly
`rel_m −12` — the edge of v6's window. Widening it is only meaningful with a control, because
the attriters who *have* 18 months of pre-event history are a later, longer-tenured cohort than
the full set. The same restricted cohort is therefore re-read at the v6 window: if a feature
moves from −12 to −16 **on the same customers**, that is an onset. If it pins at −18 again, the
feature has no onset in this panel and belongs with the standing marker (§11d), not with the
early-warning signals.

In [ ]:
# =====================================================================
# 3 · THE WIDENED PRE-WINDOW, WITH A CONTROL             [OUTPUT BLOCK 2]
# =====================================================================
# TRAP being avoided (locatability v1, defect 5): an error-vs-k curve
# read across a drifting cohort measures composition, not effect. Here
# the drift is in event month - only attriters with an event late enough
# to have 18 months of dd behind it can enter, and those are 2025-2026
# events. So the widened cohort is read TWICE: once at -18 and once at
# the v6 window, and only the WITHIN-COHORT movement is evidence.
RULE = {s["feature"]: s["rule"] for s in SIGNALS}

def col_for(f):
    r = RULE.get(f, "dd")
    return (("rate_any", SEP_RATE) if r == "zero" else
            ("rate_neg", SEP_RATE) if r == "sign" else ("med_dd", SEP_LEVEL))

def coverage_of(cur, f):
    g = cur[cur.feature == f]
    col, _ = col_for(f)
    num = g.n_val.sum() if col in ("rate_any", "rate_neg") else g.n_dd.sum()
    return num/max(g.n.sum(), 1)

def seps(cur, search_from):
    rows = []
    for f, g in cur.groupby("feature"):
        col, thr = col_for(f)
        wv = (g.pivot_table(index="rel_m", columns="cohort", values=col)
                .reindex(columns=["attriter", "stayer"]).dropna().sort_index())
        if wv.empty: continue
        gap = (wv.attriter - wv.stayer).abs(); s = gap[gap.index >= search_from]
        sep, run = None, 0
        for rm, v in s.items():
            run = run + 1 if v > thr else 0
            if run >= HOLD: sep = rm - HOLD + 1; break
        cvg = coverage_of(cur, f)
        rows.append(dict(feature=f, read_on=col, coverage=round(cvg, 3),
                         eligible=cvg >= MIN_COVERAGE, sep_rel_m=sep,
                         at_window_edge=(sep is not None and sep <= search_from),
                         max_gap=round(s.max(), 3) if len(s) else np.nan))
    return pd.DataFrame(rows).sort_values(["eligible", "sep_rel_m", "max_gap"],
                                          ascending=[False, True, False], na_position="last")

def cohorts(defn, min_pre, seed=SEED):
    """Attriters with at least min_pre months of DD-BEARING history before
    the event, plus stayers given pseudo-events drawn from the same event
    months and held to the same history requirement. Both sides move
    together or the comparison is composition."""
    ev = f"q_{defn}"
    a = (lab.filter(F.col(ev).isNotNull())
         .select("cust_pwr_id", F.col(ev).alias("event_m"),
                 F.coalesce("first_live_m", "first_m").alias("first_m"), "last_m")
         .filter(F.col("event_m") - F.greatest(F.col("first_m"), F.lit(DD_FIRST_M)) >= min_pre)
         .withColumn("cohort", F.lit("attriter")))
    dr = [r.event_m for r in a.select("event_m").limit(3000).collect()]
    dr = dr[::max(1, len(dr)//300)][:300] or [min_pre + DD_FIRST_M]
    arr = F.array(*[F.lit(int(x)) for x in dr])
    s = (lab.filter(F.col("q_A_full_exit").isNull() & F.col("q_B_bal_exit").isNull())
         .withColumn("event_m", F.element_at(
             arr, (F.abs(F.hash(F.concat_ws("|", F.col("cust_pwr_id"), F.lit(seed)))) % len(dr)) + 1))
         .select("cust_pwr_id", "event_m", F.coalesce("first_live_m", "first_m").alias("first_m"), "last_m")
         .filter(F.col("event_m") - F.greatest(F.col("first_m"), F.lit(DD_FIRST_M)) >= min_pre)
         .withColumn("cohort", F.lit("stayer")))
    return a.unionByName(s).filter(F.col("last_m") >= F.col("event_m"))

def curves(co, pre, post=EVENT_POST):
    es = (long.join(co, "cust_pwr_id", "inner")
          .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
          .filter(F.col("rel_m").between(-pre, post)))
    cur = (es.groupBy("feature", "cohort", "rel_m").agg(
               F.count("*").alias("n"),
               F.sum(F.col("dd").isNotNull().cast("int")).alias("n_dd"),
               F.sum(F.col("value").isNotNull().cast("int")).alias("n_val"),
               F.expr("percentile_approx(dd, 0.5)").alias("med_dd"),
               F.avg(F.when(F.col("value").isNotNull(), (F.col("value") > 0).cast("double"))).alias("rate_any"),
               F.avg(F.when(F.col("value").isNotNull(), (F.col("value") < 0).cast("double"))).alias("rate_neg"))
           ).toPandas()
    cur.loc[cur.n < MIN_CELL_N, ["med_dd", "rate_any", "rate_neg"]] = np.nan
    return es, cur

CO_W  = cohorts(PRIMARY_DEF, WIDE_PRE).persist(StorageLevel.DISK_ONLY)
CO_F  = cohorts(PRIMARY_DEF, EVENT_PRE).persist(StorageLevel.DISK_ONLY)
n_wa, n_ws = CO_W.filter("cohort='attriter'").count(), CO_W.filter("cohort='stayer'").count()
n_fa       = CO_F.filter("cohort='attriter'").count()

_, CUR_W18 = curves(CO_W, WIDE_PRE)            # widened window, restricted cohort
_, CUR_W12 = curves(CO_W, EVENT_PRE)           # v6 window,      SAME cohort  <- the control
ES_F, CUR_F = curves(CO_F, EVENT_PRE)          # v6 window,      full cohort
ES_F = ES_F.persist(StorageLevel.DISK_ONLY)
for nm, cu in [("wide18", CUR_W18), ("wide_ctrl12", CUR_W12), ("full12", CUR_F)]:
    cu.to_csv(OUT_DIR / f"v7_curves_{nm}.csv", index=False)

S18, S12C, S12F = seps(CUR_W18, SEARCH_FROM_WIDE), seps(CUR_W12, -EVENT_PRE), seps(CUR_F, -EVENT_PRE)
CMP = (S12F[["feature", "coverage", "eligible", "sep_rel_m"]]
       .rename(columns={"sep_rel_m": "v6_full_cohort"})
       .merge(S12C[["feature", "sep_rel_m"]].rename(columns={"sep_rel_m": "same_cohort_12"}), on="feature", how="outer")
       .merge(S18[["feature", "sep_rel_m", "at_window_edge"]]
              .rename(columns={"sep_rel_m": "widened_18", "at_window_edge": "still_at_edge"}),
              on="feature", how="outer"))
CMP["onset_found"] = np.where(CMP.still_at_edge.fillna(False).astype(bool), "NO — pinned at -18 too",
                      np.where(CMP.widened_18 < CMP.same_cohort_12, "yes", "no change"))
CMP["cohort_effect"] = CMP.same_cohort_12 - CMP.v6_full_cohort
CMP = CMP.sort_values(["eligible", "widened_18"], ascending=[False, True], na_position="last")
disp(CMP, title="2a &middot; v6 window on the full cohort · the SAME window on the restricted "
                "cohort · the widened window. <b>cohort_effect isolates how much of any movement "
                "is composition</b>", n=40, save="v7_wide_window")

WALL = CMP[CMP.feature.isin(WALL_FEATS)]
disp(WALL, title="2b &middot; The four that hit exactly &minus;12 in v6 — the reason this block exists",
     save="v7_wall_features")
kv([("attriters with >=%d months of dd history" % WIDE_PRE, n_wa),
    ("...as a share of the %d-month cohort" % EVENT_PRE, round(pct(n_wa, n_fa), 3)),
    ("matched stayers", n_ws),
    ("of the four, onset now visible", int((WALL.onset_found == "yes").sum())),
    ("of the four, still pinned at the edge", int(WALL.still_at_edge.fillna(False).sum())),
    ("median cohort_effect across eligible features", round(float(
        CMP.loc[CMP.eligible == True, "cohort_effect"].median()), 2))],
   title="2c &middot; Reading 2a", save="v7_wide_headline")
note("WALL", "Do the four -12 signals have a visible onset?",
     f"{int((WALL.onset_found=='yes').sum())} of 4 move earlier on the same cohort at rel_m -18",
     "A feature that pins at the window edge twice has no onset in this panel and is a standing "
     "marker, not a trigger. cohort_effect is the honest control: the restricted cohort is later "
     "and longer-tenured, so part of any movement is composition, not signal.")


## 3 · The calendar-time risk set

One row per (customer, month) while the customer has a live account and has not yet had the
event. Wide `dd`, a missingness indicator beside every `dd`, and the raw values for the three
features that are read on a rate rather than a ratio. Forward labels are derived in pandas from
`event_m`, so both definitions and all three horizons come out of a **single Spark pass**.

Two rules the build enforces and the output reports on:

- **Nothing at or after `t` enters a feature.** `dd_t` uses `value_t` against a reference window
  `t−6…t−4`. The peer median is cross-sectional in the same calendar month, which is available
  in production at `t`. The frozen decile comes from the customer's first three months.
- **A label must be observable.** `y_H(t)` is only defined where `t + H` is inside the panel, or
  where the event has already happened inside the window. Rows failing that are dropped and
  counted rather than filled with a zero, which would quietly relabel every censored customer as
  a stayer.

In [ ]:
# =====================================================================
# 4 · CALENDAR-TIME RISK SET                             [OUTPUT BLOCK 3]
# =====================================================================
t0 = time.time()
# Multi-agg pivot names its output {pivotValue}_{aggAlias} - the same rule
# that produced the fin_out_new_out AnalysisException in v2 (02 §6). Here
# it is used deliberately: bal_live_dd, bal_live_v, and so on.
wide = (long.groupBy("cust_pwr_id", "m_idx", "ym")
        .pivot("feature", FEATS)
        .agg(F.first("dd", True).alias("dd"), F.first("value", True).alias("v")))

RATE_RAW = ["cpty_new_out", "fin_new_out", "net_flow"]
sel = [F.col("cust_pwr_id"), F.col("m_idx"), F.col("ym")]
for f in FEATS:
    dd = F.col(f"{f}_dd")
    # log-dd: symmetric around 0, clipped so one 400x ratio cannot own a
    # Newton step. Missing stays missing here and is filled in pandas,
    # beside an indicator - never filled silently.
    sel.append(F.when(dd.isNotNull(),
                      F.log(F.greatest(F.least(dd, F.lit(DD_CLIP[1])), F.lit(DD_CLIP[0]))))
                .alias(f"ld_{f}"))
    sel.append(F.when(dd.isNull(), F.lit(1.0)).otherwise(F.lit(0.0)).alias(f"md_{f}"))
for f in RATE_RAW:
    if f"{f}_v" in wide.columns: sel.append(F.col(f"{f}_v").alias(f"raw_{f}"))
wide = wide.select(*sel)

# ── at risk = live account, pre-event for at least one definition ─────
# greatest() skips nulls in Spark, so a customer with only an A event
# keeps every month before it, and a never-event customer keeps all.
risk = (cust_month.select("cust_pwr_id", "m_idx", "ym", "bal_live", "n_accts_live")
        .filter(F.col("n_accts_live") > 0)
        .join(lab.select("cust_pwr_id",
                         F.col("q_A_full_exit").alias("event_A"),
                         F.col("q_B_bal_exit").alias("event_B"),
                         F.col("last_m")), "cust_pwr_id", "inner")
        .filter(F.col("m_idx") >= DD_FIRST_M)
        .filter(F.col("m_idx") < F.coalesce(F.greatest("event_A", "event_B"), F.lit(10**6)))
        .join(wide.drop("ym"), ["cust_pwr_id", "m_idx"], "left")
        .join(p30_month, ["cust_pwr_id", "m_idx"], "left")
        .withColumn("p30_fire", F.coalesce("p30_fire", F.lit(0))))
risk.write.mode("overwrite").partitionBy("m_idx").parquet(hp("risk_set"))
risk = spark.read.parquet(hp("risk_set")).persist(StorageLevel.DISK_ONLY)

ORIGINS = list(range(ORIGIN_START, M_MAX - PRIMARY_H + 1))
shape = (risk.groupBy("m_idx").agg(F.count("*").alias("at_risk"),
            F.avg(F.col("ld_fin_out_n").isNotNull().cast("double")).alias("cov_fin_out_n"),
            F.avg(F.col("ld_bal_live").isNotNull().cast("double")).alias("cov_bal_live"),
            F.avg("p30_fire").alias("p30_fire_rate")).orderBy("m_idx"))
SHAPE = disp(shape, title="3a &middot; The book, month by month. <b>cov_* is the share of the "
                          "at-risk book that feature can score at all</b> — the number no v6 "
                          "table carries", n=40, save="v7_risk_shape")

# ── the collects ──────────────────────────────────────────────────────
MODEL_COLS = (["cust_pwr_id", "m_idx", "ym", "event_A", "event_B", "last_m",
               "p30_fire", "bal_live", "n_accts_live"]
              + [f"ld_{f}" for f in FEATS] + [f"md_{f}" for f in FEATS]
              + [f"raw_{f}" for f in RATE_RAW if f"raw_{f}" in risk.columns])
H_MAX = max(HORIZONS)
is_pos = (F.col("event_A").between(F.col("m_idx")+1, F.col("m_idx")+H_MAX) |
          F.col("event_B").between(F.col("m_idx")+1, F.col("m_idx")+H_MAX))
# TRAIN keeps every positive and samples negatives at NEG_SAMPLE. The
# intercept is corrected by ln(NEG_SAMPLE) so predicted probabilities -
# and therefore precision - are on the population scale. Ranking is
# unaffected either way; precision is not, and precision is the output.
train_s = (risk.filter(F.col("m_idx") <= max(ORIGINS) - min(HORIZONS))
           .withColumn("_pos", is_pos.cast("int"))
           .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                                                       F.col("m_idx").cast("string"),
                                                       F.lit(SEED)))) % 100000)/100000.0)
           .filter((F.col("_pos") == 1) | (F.col("_u") < NEG_SAMPLE))
           .select(*MODEL_COLS))
TRAIN_RAW = collect_pd(train_s, "TRAIN (negatives sampled)")

TEST_RAW = {}
for Tm in ORIGINS:
    q = risk.filter(F.col("m_idx") == Tm)
    if TEST_STAYER_FRAC < 1.0:
        q = q.filter(is_pos | ((F.abs(F.hash(F.concat_ws("|", "cust_pwr_id", F.lit(SEED+1))))
                                % 100000)/100000.0 < TEST_STAYER_FRAC))
    TEST_RAW[Tm] = collect_pd(q.select(*MODEL_COLS), f"TEST m_idx={Tm}")

kv([("at-risk customer-months", int(SHAPE.at_risk.sum())),
    ("test origins", f"m_idx {ORIGINS[0]}–{ORIGINS[-1]} ({len(ORIGINS)} folds)"),
    ("train rows collected", len(TRAIN_RAW)),
    ("test rows collected", int(sum(len(v) for v in TEST_RAW.values()))),
    ("negative sampling in TRAIN", NEG_SAMPLE),
    ("stayer sampling in TEST", TEST_STAYER_FRAC),
    ("block 3 wall (s)", round(time.time()-t0))],
   title="3b &middot; Collects", save="v7_collects")


In [ ]:
# =====================================================================
# 4b · FEATURE PREP + LABELS IN PANDAS
# =====================================================================
LD  = [f"ld_{f}" for f in FEATS]
MD  = [f"md_{f}" for f in FEATS]
RAW = [f"raw_{f}" for f in RATE_RAW if f"raw_{f}" in TRAIN_RAW.columns]

def prep(df):
    """ld NaN -> 0 (dd = 1, i.e. moving exactly with peers) with md
    carrying the fact that it was absent. That pairing is the whole
    argument of §4: the model can score a customer no single dd feature
    can, because 'this customer has no measurable institution activity'
    is itself evidence."""
    d = df.copy()
    for c in LD: d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
    for c in MD: d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
    def _raw(name):
        return (pd.to_numeric(d[name], errors="coerce") if name in d.columns
                else pd.Series(np.nan, index=d.index, dtype="float64"))
    for f, z in [("cpty_new_out", "z_cpty_new"), ("fin_new_out", "z_fin_new")]:
        r = _raw(f"raw_{f}")
        d[z]     = (r <= 0).astype(float).where(r.notna(), 0.0)
        d["m"+z] = r.isna().astype(float)
    r = _raw("raw_net_flow")
    d["z_negflow"]  = (r < 0).astype(float).where(r.notna(), 0.0)
    d["mz_negflow"] = r.isna().astype(float)
    return d

def label(d, defn, H):
    """y = 1 if the event lands in (t, t+H]. A row whose panel ends before
    t+H and which has no event is UNOBSERVABLE, not a zero - dropping it
    is the difference between right-censoring and quietly relabelling
    every censored customer a stayer."""
    ev = pd.to_numeric(d["event_A" if defn == "A_full_exit" else "event_B"], errors="coerce")
    t  = pd.to_numeric(d["m_idx"], errors="coerce")
    y  = ((ev > t) & (ev <= t + H)).astype(float)
    observable = (t + H <= M_MAX) | (y == 1)
    out = d.loc[observable].copy(); out["y"] = y.loc[observable].values
    return out

TRAIN_P = prep(TRAIN_RAW)
TEST_P  = {k: prep(v) for k, v in TEST_RAW.items()}
Z_COLS  = ["z_cpty_new", "z_fin_new", "z_negflow", "mz_cpty_new", "mz_fin_new", "mz_negflow"]

def cols_for(feats, rates=True):
    c = [f"ld_{f}" for f in feats] + [f"md_{f}" for f in feats]
    return c + Z_COLS if rates else c

PAY_FEATS = [f for f in FEATS if f != "bal_live"]
SPECS = {
    "M1_bal_only":     cols_for(["bal_live"], rates=False),
    "M2_fin_only":     cols_for(["fin_out_n"], rates=False),
    "M3_pay_only":     cols_for(PAY_FEATS, rates=True),
    "M4_pay_plus_bal": cols_for(FEATS,      rates=True),
}

# ── the un-fitted baselines, exactly as v6 defined them ───────────────
_LOGT = math.log(SCORE_THRESH)
SIG12 = [s["feature"] for s in SIGNALS]
def count_rule(d):
    """v6 §5g's unweighted count. Every signal weighs the same, so a 30x
    feature and a 1.1x feature are one vote each. Needs >=6 evaluable
    signals or the row is unscoreable, same guard as v6."""
    fired = pd.DataFrame(index=d.index); evalu = pd.DataFrame(index=d.index)
    for f in SIG12:
        if RULE.get(f) == "zero":
            z = "z_cpty_new" if f == "cpty_new_out" else "z_fin_new"
            fired[f] = d[z]; evalu[f] = 1.0 - d["m"+z]
        elif RULE.get(f) == "sign":
            fired[f] = d["z_negflow"]; evalu[f] = 1.0 - d["mz_negflow"]
        else:
            fired[f] = ((d[f"ld_{f}"] < _LOGT) & (d[f"md_{f}"] == 0)).astype(float)
            evalu[f] = 1.0 - d[f"md_{f}"]
    k = fired.sum(axis=1); m = evalu.sum(axis=1)
    return np.where(m >= 6, k, np.nan)

def rule_scores(d):
    """Higher = riskier for every score, so the single-feature rules are
    NEGATED log-dd. Unscoreable stays NaN and is counted, never imputed."""
    out = {}
    out["R_fin_out_n"] = np.where(d["md_fin_out_n"] == 0, -d["ld_fin_out_n"], np.nan)
    out["R_bal_live"]  = np.where(d["md_bal_live"]  == 0, -d["ld_bal_live"],  np.nan)
    out["R_count12"]   = count_rule(d)
    out["R_p30"]       = pd.to_numeric(d["p30_fire"], errors="coerce").astype(float).values
    return out
print(f"{len(SPECS)} fitted specs · 4 rule baselines · {len(LD)} dd features")


## 4 · The fair test of "combining doesn't help"

v6's verdict — single beats combined in 11 of 12 months — is a real result **about an unweighted
count of twelve equally-weighted votes**, and the handoff already says so. This is the weighted
test it asked for.

Rolling origin. For test month `T` and horizon `H`, training uses only rows with `t ≤ T − H`, so
every training label has already resolved by `T`. Five things are compared on the same rows:
balance alone, `fin_out_n` alone, v6's count, payments-only, and payments plus balance. AUC is
computed on scoreable rows only; **queue coverage is reported beside it**, because a model with
AUC 0.80 on 35% of the book is not comparable to one with AUC 0.78 on all of it.

In [ ]:
# =====================================================================
# 5 · ROLLING-ORIGIN DISCRETE-TIME HAZARD                [OUTPUT BLOCK 4]
# =====================================================================
def run_folds(defn, H, capacity=CAPACITY):
    rows, tk, fits, scored = [], [], {}, {}
    for Tm in ORIGINS:
        tr = label(TRAIN_P[TRAIN_P.m_idx <= Tm - H], defn, H)
        te = label(TEST_P[Tm], defn, H)
        if tr["y"].sum() < MIN_TRAIN_POS or te["y"].sum() < 1 or te.empty:
            continue
        sc = rule_scores(te)
        for nm, cols in SPECS.items():
            sp = fit_spec(tr, cols, NEG_SAMPLE)
            fits[(defn, H, Tm, nm)] = sp
            sc[nm] = apply_spec(sp, te)
        p_base = float(te["y"].mean())
        scored[Tm] = te[["cust_pwr_id", "m_idx", "y", "event_A", "event_B", "p30_fire"]].copy()
        for nm, s in sc.items():
            scored[Tm][nm] = s
            rows.append(dict(defn=defn, H=H, origin=Tm, model=nm, n_at_risk=len(te),
                             n_pos=int(te["y"].sum()), base_rate=p_base,
                             auc=auc(te["y"].values, s),
                             queue_coverage=float(np.isfinite(np.asarray(s, dtype=float)).mean()),
                             n_train=len(tr), n_train_pos=int(tr["y"].sum())))
            m = topk_metrics(s, te["y"].values, capacity, p_base=p_base)
            m.insert(0, "model", nm); m.insert(0, "origin", Tm)
            m.insert(0, "H", H);      m.insert(0, "defn", defn)
            tk.append(m)
    return (pd.DataFrame(rows),
            pd.concat(tk, ignore_index=True) if tk else pd.DataFrame(),
            fits, scored)

t0 = time.time()
FOLD, TOPK, FITS, SCORED = {}, {}, {}, {}
for H in HORIZONS:
    f, k, ft, sc = run_folds(PRIMARY_DEF, H)
    FOLD[(PRIMARY_DEF, H)], TOPK[(PRIMARY_DEF, H)] = f, k
    FITS.update(ft)
    if H == PRIMARY_H: SCORED[PRIMARY_DEF] = sc
print(f"  folds fitted in {time.time()-t0:,.0f}s")

ALLF = pd.concat([FOLD[k] for k in FOLD], ignore_index=True)
ALLK = pd.concat([TOPK[k] for k in TOPK], ignore_index=True)
ALLF.to_csv(OUT_DIR / "v7_folds_A.csv", index=False)
ALLK.to_csv(OUT_DIR / "v7_topk_A.csv", index=False)

AUCT = (ALLF.pivot_table(index="model", columns="H", values="auc", aggfunc="mean")
        .join(ALLF.groupby("model").queue_coverage.mean().rename("queue_coverage"))
        .join(ALLF.groupby("model").auc.std().rename("auc_sd_across_folds"))
        .sort_values(PRIMARY_H, ascending=False).round(4).reset_index())
disp(AUCT, title="4a &middot; Mean AUC across rolling origins, by horizon. <b>Read it beside "
                 "queue_coverage</b> — an AUC on a third of the book is not the same object as "
                 "an AUC on all of it", save="v7_auc")

# Pooled top-K: total true positives over total alerts across folds, not
# a mean of per-fold precisions. A mean of ratios over months with
# different at-risk counts is not the precision anyone experiences.
PK = (ALLK[ALLK.H == PRIMARY_H].groupby(["model", "k"], as_index=False)
      .agg(tp=("tp", "sum"), alerts=("k", "sum"), pos=("n_at_risk", "size"),
           recall=("recall", "mean"), coverage=("queue_coverage", "mean")))
PK["precision"] = PK.tp/PK.alerts
_b = ALLF[ALLF.H == PRIMARY_H].base_rate.mean()
PK["lift"] = PK.precision/_b
PK["alerts_per_tp"] = PK.alerts/PK.tp.replace(0, np.nan)
disp(PK.pivot_table(index="model", columns="k", values="precision").round(4).reset_index(),
     title=f"4b &middot; POOLED precision at K alerts a month, horizon {PRIMARY_H}m "
           f"(base rate {_b:.4%})", save="v7_pooled_precision")
disp(PK.pivot_table(index="model", columns="k", values="lift").round(1).reset_index(),
     title="4c &middot; The same as lift", save="v7_pooled_lift")
disp(PK.pivot_table(index="model", columns="k", values="recall").round(3).reset_index(),
     title="4d &middot; Recall — the trade against 4b", save="v7_pooled_recall")

# R_p30 is binary, so a top-K rank over it is arbitrary among the fired.
# It is measured properly as a whole flagged set in 5b and excluded here.
PKc  = PK[PK.model != "R_p30"]
_w   = PKc[PKc.k == QUEUE_K].set_index("model")
_best = str(_w.precision.idxmax())
_fin, _cnt, _bal = "R_fin_out_n", "R_count12", "R_bal_live"
kv([("best model at K=%d" % QUEUE_K, _best),
    ("  ...its precision", round(float(_w.loc[_best, "precision"]), 4)),
    ("  ...its lift", round(float(_w.loc[_best, "lift"]), 1)),
    ("  ...its queue coverage", round(float(_w.loc[_best, "coverage"]), 3)),
    ("fin_out_n rule precision", round(float(_w.loc[_fin, "precision"]), 4) if _fin in _w.index else None),
    ("fin_out_n rule queue coverage", round(float(_w.loc[_fin, "coverage"]), 3) if _fin in _w.index else None),
    ("v6 unweighted count precision", round(float(_w.loc[_cnt, "precision"]), 4) if _cnt in _w.index else None),
    ("balance-only rule precision", round(float(_w.loc[_bal, "precision"]), 4) if _bal in _w.index else None),
    ("does a WEIGHTED model beat every single-feature rule?",
     "yes" if _best in ("M3_pay_only", "M4_pay_plus_bal") else "no")],
   title="4e &middot; §4.2 answered on weights, not on an unweighted count", save="v7_model_verdict")

# ── 4f · coverage or discrimination? ──────────────────────────────────
# A fitted model scores 100% of the book by construction, so its AUC is
# over a different denominator than a rule's. Re-scoring everything on
# the subset where fin_out_n IS defined separates the two claims: does
# the model discriminate better, or does it merely reach further?
_S   = pd.concat(SCORED[PRIMARY_DEF].values(), ignore_index=True)
_msk = np.isfinite(pd.to_numeric(_S["R_fin_out_n"], errors="coerce").to_numpy())
_sub = _S[_msk]
SUBA = pd.DataFrame([dict(
        model=m,
        auc_whole_book=auc(_S["y"].values, pd.to_numeric(_S[m], errors="coerce").values),
        auc_where_fin_defined=auc(_sub["y"].values, pd.to_numeric(_sub[m], errors="coerce").values),
        n_whole=len(_S), n_subset=int(_msk.sum()))
    for m in [c for c in list(SPECS) + ["R_fin_out_n", "R_bal_live", "R_count12"] if c in _S.columns]])
SUBA["reach_premium"] = (SUBA.auc_whole_book - SUBA.auc_where_fin_defined).round(4)
disp(SUBA.round(4).sort_values("auc_where_fin_defined", ascending=False),
     title="4f &middot; Coverage or discrimination? Left column is the whole book, right column "
           "is only where <code>fin_out_n</code> has a dd. <b>If the model wins on the left but "
           "ties on the right, what it buys is reach, not sharper ranking</b> — which is still "
           "the reason to ship it, and should be said in those words",
     save="v7_coverage_vs_discrimination")

INC = (AUCT.set_index("model")[PRIMARY_H].to_dict() if PRIMARY_H in AUCT.columns
       else AUCT.set_index("model").iloc[:, 0].to_dict())
note("MODEL", "Does combining signals help once they are WEIGHTED?",
     f"best AUC {max(INC, key=INC.get)} at {max(INC.values()):.4f}",
     "v6 proved only that an unweighted count of twelve equal votes loses to the best single "
     "feature. This is a fitted discrete-time hazard on a rolling origin. The decisive column is "
     "queue_coverage: fin_out_n cannot score two thirds of the book, and a model carrying "
     "missingness indicators can.")
note("INCR", "Does payment behaviour add over the deposit balance?",
     f"M4 {INC.get('M4_pay_plus_bal', float('nan')):.4f} vs M1 balance-only "
     f"{INC.get('M1_bal_only', float('nan')):.4f}",
     "Measured on the same rows, same folds, no counterparty data. This is the number that "
     "should be re-run unchanged when PAYS_CPTY lands.")


## 5 · The queue

§4.3 of the handoff: *nobody has priced the precision/recall trade.* This block prices it as a
capacity decision, which is the form the decision actually takes — a sales team works a list of a
fixed size every month, not a threshold.

Four things get measured that no earlier run measured at all:

1. **Queue coverage.** What share of the at-risk book can each rule even score.
2. **Lead.** Of the attriters the queue catches, how many months before the exit it first caught
   them — the number that competes with the incumbent's median of 2.
3. **Alert fatigue.** A customer in the top 250 for six months running is one relationship
   conversation, not six. Alerts-per-true-positive is reported both ways.
4. **Two tiers versus one longer list.** The handoff proposes `fin_out_n` over `bal_live`. The
   honest comparison is against simply taking 1,250 names off a single ranked list, and that
   comparison has not been run.

In [ ]:
# =====================================================================
# 6 · THE QUEUE                                          [OUTPUT BLOCK 5]
# =====================================================================
S = pd.concat(SCORED[PRIMARY_DEF].values(), ignore_index=True)
S["event_m"] = pd.to_numeric(S["event_A"], errors="coerce")
MODELS = [c for c in list(SPECS) + ["R_fin_out_n", "R_bal_live", "R_count12"] if c in S.columns]
BEST_MODEL = str(PKc[PKc.k == QUEUE_K].set_index("model").precision.idxmax())

def flagged(df, model, K):
    d = df.copy()
    d["_s"] = pd.to_numeric(d[model], errors="coerce")
    d["_r"] = d.groupby("m_idx")["_s"].rank(ascending=False, method="first", na_option="bottom")
    return d[d._r <= K]

def fatigue(fl):
    """Two counts. Raw alerts is what the queue emits; distinct
    conversations collapses re-flags inside COOLDOWN_M, because a name
    that reappears next month is the same conversation. Every
    alerts-per-TP figure in this programme so far has been the first
    number quoted as if it were the second."""
    if fl.empty: return 0, 0, 0
    g = fl.sort_values(["cust_pwr_id", "m_idx"])
    gap = g.groupby("cust_pwr_id")["m_idx"].diff()
    new_convo = gap.isna() | (gap > COOLDOWN_M)
    return len(fl), int(g.cust_pwr_id.nunique()), int(new_convo.sum())

def lead_profile(fl):
    """First month the queue caught an attriter, against its event month."""
    a = fl[fl.event_m.notna()].copy()
    if a.empty: return pd.Series(dtype=float)
    first = a.groupby("cust_pwr_id").agg(first_flag=("m_idx", "min"), event_m=("event_m", "max"))
    return (first.event_m - first.first_flag).astype(float)

rows = []
for m in MODELS:
    for K in CAPACITY:
        fl = flagged(S, m, K)
        tp = int(fl.y.sum()); raw, dis, con = fatigue(fl)
        ld = lead_profile(fl[fl.y == 1])
        rows.append(dict(model=m, k=K, alerts_raw=raw, distinct_customers=dis,
                         conversations=con, tp=tp,
                         precision=tp/max(raw, 1),
                         alerts_per_tp=raw/tp if tp else np.nan,
                         conversations_per_tp=con/tp if tp else np.nan,
                         median_lead_m=float(ld.median()) if len(ld) else np.nan,
                         p90_lead_m=float(ld.quantile(.9)) if len(ld) else np.nan))
QUEUE = pd.DataFrame(rows)
QUEUE.to_csv(OUT_DIR / "v7_queue_grid.csv", index=False)
disp(QUEUE[QUEUE.k.isin([100, QUEUE_K, 1000])].sort_values(["k", "precision"], ascending=[True, False]),
     title="5a &middot; The queue at three capacities. <b>conversations_per_tp is the honest cost "
           "figure</b>; alerts_per_tp double-counts a name that reappears next month",
     n=40, save="v7_queue_headline")

# ── the incumbent in the SAME calendar frame ──────────────────────────
inc = S[S.p30_fire == 1]
i_raw, i_dis, i_con = fatigue(inc)
i_tp = int(inc.y.sum())
i_ld = lead_profile(inc[inc.y == 1])
_bm = QUEUE[(QUEUE.model == BEST_MODEL) & (QUEUE.k == QUEUE_K)].iloc[0]
disp(pd.DataFrame([
    dict(rule="incumbent 30% balance rule", alerts_per_month=round(i_raw/len(ORIGINS)),
         alerts_raw=i_raw, distinct_customers=i_dis, conversations=i_con, tp=i_tp,
         precision=i_tp/max(i_raw, 1), conversations_per_tp=i_con/max(i_tp, 1),
         median_lead_m=float(i_ld.median()) if len(i_ld) else np.nan,
         recall=i_tp/max(int(S.y.sum()), 1)),
    dict(rule=f"{BEST_MODEL} at K={QUEUE_K}", alerts_per_month=QUEUE_K,
         alerts_raw=int(_bm.alerts_raw), distinct_customers=int(_bm.distinct_customers),
         conversations=int(_bm.conversations), tp=int(_bm.tp), precision=float(_bm.precision),
         conversations_per_tp=float(_bm.conversations_per_tp),
         median_lead_m=float(_bm.median_lead_m), recall=_bm.tp/max(int(S.y.sum()), 1))]),
     title="5b &middot; The incumbent, measured the way a queue is measured. v6 gave it a "
           "<i>lead time</i>; this is its <i>cost</i>", save="v7_incumbent_queue")

# ── two tiers vs one longer list ──────────────────────────────────────
t1 = flagged(S, BEST_MODEL, TIER1_K)
key1 = set(zip(t1.cust_pwr_id, t1.m_idx))
t2all = flagged(S, "R_bal_live", TIER1_K + TIER2_K)
_dup  = pd.MultiIndex.from_arrays([t2all.cust_pwr_id, t2all.m_idx]).isin(key1)
t2    = t2all[~_dup].groupby("m_idx", group_keys=False).head(TIER2_K)
one = flagged(S, BEST_MODEL, TIER1_K + TIER2_K)
ov  = flagged(S, "R_fin_out_n", QUEUE_K)
ovb = flagged(S, "R_bal_live", QUEUE_K)
_a = set(zip(ov.cust_pwr_id, ov.m_idx)); _b = set(zip(ovb.cust_pwr_id, ovb.m_idx))
disp(pd.DataFrame([
    dict(design="tier 1 only (model)", alerts=len(t1), tp=int(t1.y.sum()),
         precision=t1.y.sum()/max(len(t1), 1), recall=t1.y.sum()/max(int(S.y.sum()), 1)),
    dict(design="tier 2 only (balance, net of tier 1)", alerts=len(t2), tp=int(t2.y.sum()),
         precision=t2.y.sum()/max(len(t2), 1), recall=t2.y.sum()/max(int(S.y.sum()), 1)),
    dict(design="two tiers combined", alerts=len(t1)+len(t2), tp=int(t1.y.sum()+t2.y.sum()),
         precision=(t1.y.sum()+t2.y.sum())/max(len(t1)+len(t2), 1),
         recall=(t1.y.sum()+t2.y.sum())/max(int(S.y.sum()), 1)),
    dict(design=f"ONE list, {TIER1_K+TIER2_K} deep, same model", alerts=len(one), tp=int(one.y.sum()),
         precision=one.y.sum()/max(len(one), 1), recall=one.y.sum()/max(int(S.y.sum()), 1))]),
     title="5c &middot; §5.1's two-tier proposal against the obvious alternative — one ranked "
           "list, taken deeper. <b>If the single list wins, the two-tier design is complexity "
           "with no payoff</b>", save="v7_two_tier")
kv([("top-%d overlap, fin_out_n vs bal_live" % QUEUE_K,
     round(len(_a & _b)/max(len(_a | _b), 1), 3)),
    ("names only fin_out_n finds", len(_a - _b)),
    ("names only bal_live finds", len(_b - _a)),
    ("best model at K", BEST_MODEL),
    ("its median lead (months)", _bm.median_lead_m),
    ("incumbent median lead (months)", float(i_ld.median()) if len(i_ld) else np.nan),
    ("its conversations per true positive", round(float(_bm.conversations_per_tp), 1)),
    ("incumbent conversations per true positive", round(i_con/max(i_tp, 1), 1))],
   title="5d &middot; Reading 5a&ndash;5c", save="v7_queue_verdict")
note("QUEUE", "What does the queue actually deliver at 250 alerts a month?",
     f"precision {float(_bm.precision):.3f}, median lead {_bm.median_lead_m:.0f}m, "
     f"{float(_bm.conversations_per_tp):.1f} conversations per true positive",
     "Compared against the incumbent in the same calendar frame rather than as a lead time. "
     "Two tiers are only worth building if 5c shows them beating one list taken deeper.")


## 6 · `B_bal_exit` — does the queue survive the shell-account population

Handoff §5.4. `B` customers keep their accounts open and let the balance go to zero, so their
balance separates *later* (−3 versus −5) and the payment lead should be wider. Same risk set,
same folds, labels re-derived from `event_B`.

In [ ]:
# =====================================================================
# 7 · B_bal_exit VALIDATION                              [OUTPUT BLOCK 6]
# =====================================================================
fB, kB, ftB, scB = run_folds("B_bal_exit", PRIMARY_H)
fB.to_csv(OUT_DIR / "v7_folds_B.csv", index=False); kB.to_csv(OUT_DIR / "v7_topk_B.csv", index=False)
FITS.update(ftB); SCORED["B_bal_exit"] = scB

PKB = (kB.groupby(["model", "k"], as_index=False)
       .agg(tp=("tp", "sum"), alerts=("k", "sum"), recall=("recall", "mean"),
            coverage=("queue_coverage", "mean")))
PKB["precision"] = PKB.tp/PKB.alerts
AB = (pd.concat([ALLF[ALLF.H == PRIMARY_H].assign(defn="A_full_exit"), fB], ignore_index=True)
      .pivot_table(index="model", columns="defn", values="auc", aggfunc="mean").round(4))
if {"A_full_exit", "B_bal_exit"} <= set(AB.columns):
    AB["auc_delta_B_minus_A"] = (AB["B_bal_exit"] - AB["A_full_exit"]).round(4)
disp(AB.reset_index(), title="6a &middot; AUC on both definitions, same folds", save="v7_auc_AB")

_pa = PK[PK.k == QUEUE_K].set_index("model").precision
_pb = PKB[PKB.k == QUEUE_K].set_index("model").precision
disp(pd.concat([_pa.rename("A_full_exit"), _pb.rename("B_bal_exit")], axis=1)
     .assign(ratio=lambda d: (d.B_bal_exit/d.A_full_exit).round(2)).round(4).reset_index(),
     title=f"6b &middot; Precision at K={QUEUE_K} on both. The base rates differ "
           f"(0.91% vs 0.78%), so read the ratio, not the level", save="v7_prec_AB")
_ra = AB["A_full_exit"].rank(ascending=False) if "A_full_exit" in AB.columns else None
_rb = AB["B_bal_exit"].rank(ascending=False) if "B_bal_exit" in AB.columns else None
kv([("model ordering identical on A and B",
     bool((_ra.sort_index().values == _rb.sort_index().values).all()) if _ra is not None and _rb is not None else None),
    ("best model on B", str(AB["B_bal_exit"].idxmax()) if "B_bal_exit" in AB.columns else None),
    ("best model on A", str(AB["A_full_exit"].idxmax()) if "A_full_exit" in AB.columns else None)],
   title="6c &middot; Does the queue transfer?", save="v7_AB_verdict")
note("VALB", "Does the queue work for the shell-account population?",
     f"best on B: {AB['B_bal_exit'].idxmax() if 'B_bal_exit' in AB.columns else 'n/a'}",
     "If the model ordering is the same on both definitions, one queue serves both populations "
     "and the label choice stops being a live design question.")


## 7 · Competitor or contraction — a candidate segmentation, not a classifier

Handoff §4.4. **There is no ground truth for this and nothing in this block should be briefed as
a validated split.** What is available is a discriminator that does not need counterparty data:

- If a client **moved the flow to a competitor**, the counterparties are still there. They keep
  getting paid — by other PNC customers, visibly — while our client stops paying them *through
  us*. Ticket size holds and payment count collapses, which is exactly the v6 §5c finding.
- If a client **contracted**, the counterparties go quiet everywhere and the ticket shrinks with
  the count.

`pay_pairs` supports the first axis: for each attriter's baseline counterparties, what share are
still being paid by *some other* PNC customer after the exit. It is confounded by counterparty
popularity — a processor is always alive — so it is read as a **difference against a matched
stayer baseline**, never as a level. The first cell probes the schema; the constants in §0 must
be corrected from that probe before anything below is read.

In [ ]:
# =====================================================================
# 8a · pay_pairs SCHEMA PROBE                            [OUTPUT BLOCK 7]
# =====================================================================
ATTRIB_OK = False
if RUN_ATTRIB:
    try:
        pp = spark.read.parquet(v2("pay_pairs"))
        disp(pd.DataFrame(pp.dtypes, columns=["column", "type"]), title="7a &middot; pay_pairs schema", n=30)
        need = {"cust_pwr_id", "kt", "k", "ym"}
        if not need <= set(pp.columns):
            print(f"  columns {sorted(need - set(pp.columns))} absent — "
                  f"correct the §0 PP_* constants and the joins below, then re-run")
        else:
            disp(pp.groupBy("kt").agg(F.count("*").alias("rows")).orderBy(F.desc("rows")),
                 title="7a2 &middot; kt values", n=10)
            if "flow" in pp.columns:
                disp(pp.groupBy("flow").agg(F.count("*").alias("rows")).orderBy(F.desc("rows")),
                     title="7a3 &middot; flow values", n=10)
            kts = {r[0] for r in pp.select("kt").distinct().limit(20).collect()}
            fls = ({r[0] for r in pp.select("flow").distinct().limit(20).collect()}
                   if "flow" in pp.columns else {PP_FLOW_OUT})
            ATTRIB_OK = (PP_KT_CPTY in kts) and (PP_FLOW_OUT in fls)
            print(f"  PP_KT_CPTY={PP_KT_CPTY!r} in kt: {PP_KT_CPTY in kts} · "
                  f"PP_FLOW_OUT={PP_FLOW_OUT!r} in flow: {PP_FLOW_OUT in fls}")
            if not ATTRIB_OK:
                print("  §0 constants do not match the data. 7b/7c are SKIPPED rather than run "
                      "on a wrong filter, which would return a plausible-looking empty result.")
    except Exception as e:
        print(f"  pay_pairs unavailable ({type(e).__name__}: {e}) — 7b/7c skipped")
else:
    print("  RUN_ATTRIB = False — skipped")


In [ ]:
# =====================================================================
# 8b · COUNTERPARTY LIVENESS — one Spark job per month, resumable
# =====================================================================
# Same pattern as the v2 payment feature build: a single unfiltered job
# over pay_pairs kills the SparkContext. One month, one job, skipped if
# its output already exists.
if ATTRIB_OK:
    YM_MAP = {r.ym: r.m_idx for r in cust_month.select("ym", "m_idx").distinct().collect()}
    YMS = sorted(YM_MAP)
    t0 = time.time()
    for i, y in enumerate(YMS, 1):
        out = hp(f"k_payers/ym={y}")
        try:
            spark.read.parquet(out).limit(1).count(); continue
        except Exception:
            pass
        s = time.time()
        d = pp.filter((F.col("ym") == y) & (F.col("kt") == PP_KT_CPTY))
        if "flow" in pp.columns: d = d.filter(F.col("flow") == PP_FLOW_OUT)
        (d.groupBy("k").agg(F.countDistinct("cust_pwr_id").alias("n_payers"))
           .write.mode("overwrite").parquet(out))
        print(f"  {y}  ({i}/{len(YMS)})  {time.time()-s:,.0f}s")
    # partition discovery gives ym back as a column; cast it so the join
    # key dtype cannot drift ("2024-01" is not a parseable date, so it
    # stays a string, but say so rather than rely on it)
    kp = (spark.read.option("basePath", hp("k_payers")).parquet(hp("k_payers"))
          .withColumn("ym", F.col("ym").cast("string")))
    ymdf = spark.createDataFrame([(str(k), int(v)) for k, v in YM_MAP.items()], ["ym", "m_idx"])
    kp = kp.join(ymdf, "ym", "inner").select("k", "m_idx", "n_payers")
    print(f"  k_payers ready in {time.time()-t0:,.0f}s")


In [ ]:
# =====================================================================
# 8c · SURVIVAL x TICKET-HOLD QUADRANTS
# =====================================================================
if ATTRIB_OK:
    co = CO_F.select("cust_pwr_id", "event_m", "cohort")
    ymdf2 = spark.createDataFrame([(k, int(v)) for k, v in YM_MAP.items()], ["ym", "m_idx"])
    ppc = pp.filter(F.col("kt") == PP_KT_CPTY)
    if "flow" in pp.columns: ppc = ppc.filter(F.col("flow") == PP_FLOW_OUT)
    ppc = ppc.join(ymdf2, "ym", "inner").select("cust_pwr_id", "k", "m_idx")

    # baseline counterparty set, capped: the tail is long and it is the
    # join, not the statistic, that would blow up
    base = (ppc.join(co, "cust_pwr_id", "inner")
            .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
            .filter(F.col("rel_m").between(*ATTRIB_BASE))
            .groupBy("cust_pwr_id", "cohort", "event_m", "k").agg(F.count("*").alias("n_m")))
    wk = Window.partitionBy("cust_pwr_id").orderBy(F.desc("n_m"), F.asc("k"))
    base = base.withColumn("rk", F.row_number().over(wk)).filter(F.col("rk") <= ATTRIB_MAX_CPTY)

    # is the counterparty still being paid by SOMEONE ELSE after the exit?
    post = (base.join(kp.withColumnRenamed("m_idx", "km"), "k", "inner")
            .withColumn("rel_k", F.col("km") - F.col("event_m"))
            .filter(F.col("rel_k").between(*ATTRIB_POST)))
    # our own customer is out of the book by then, so n_payers >= 1 is
    # already "someone else"; the >=2 variant is kept for the months
    # where the exit has not fully landed
    surv = (post.groupBy("cust_pwr_id", "cohort", "k")
            .agg(F.max("n_payers").alias("mx"))
            .withColumn("alive", (F.col("mx") >= 1).cast("double"))
            .groupBy("cust_pwr_id", "cohort")
            .agg(F.avg("alive").alias("cpty_survival"), F.count("*").alias("n_base_cpty"))
            .filter(F.col("n_base_cpty") >= 5))

    # ticket-hold axis, straight off the event study - no new heavy work
    tick = (ES_F.filter((F.col("feature") == "avg_ticket_out") & F.col("rel_m").between(-3, 0) &
                        F.col("dd").isNotNull())
            .groupBy("cust_pwr_id").agg(F.avg("dd").alias("ticket_hold"),
                                        F.count("*").alias("n_tick")).filter("n_tick >= 2"))
    Q = collect_pd(surv.join(tick, "cust_pwr_id", "inner"), "attribution")

    st = Q[Q.cohort == "stayer"]
    SURV_CUT   = float(st.cpty_survival.median()) if len(st) else 0.5
    TICKET_CUT = float(st.ticket_hold.median()) if len(st) else 1.0
    Q["survives"] = Q.cpty_survival >= SURV_CUT
    Q["holds"]    = Q.ticket_hold  >= TICKET_CUT
    Q["quadrant"] = np.select(
        [Q.holds & Q.survives, ~Q.holds & Q.survives, Q.holds & ~Q.survives],
        ["DISPLACEMENT — partners alive, ticket held",
         "CONTRACTION — partners alive, buying less",
         "PARTNER LOSS — partners gone, ticket held"],
        default="WIND-DOWN — partners gone, ticket gone")
    QT = (Q.pivot_table(index="quadrant", columns="cohort", values="cust_pwr_id",
                        aggfunc="count").fillna(0))
    for c in ("attriter", "stayer"):
        if c in QT.columns: QT[c+"_share"] = QT[c]/QT[c].sum()
    if {"attriter_share", "stayer_share"} <= set(QT.columns):
        QT["lift_vs_stayer"] = (QT.attriter_share/QT.stayer_share).round(2)
    disp(QT.reset_index(), title="7c &middot; CANDIDATE segmentation of attriters. The cuts are "
         f"the STAYER medians (survival {SURV_CUT:.3f}, ticket {TICKET_CUT:.3f}), so a quadrant "
         "with lift ~1.0 is not a finding", save="v7_attribution_quadrants")
    kv([("attriters classified", int((Q.cohort == 'attriter').sum())),
        ("stayers as the baseline", int((Q.cohort == 'stayer').sum())),
        ("survival cut (stayer median)", round(SURV_CUT, 3)),
        ("ticket cut (stayer median)", round(TICKET_CUT, 3)),
        ("GROUND TRUTH", "none — this is a candidate split, not a validated classifier"),
        ("validation hook", "CRM win/loss coding on a sample of the DISPLACEMENT quadrant")],
       title="7d &middot; What this is and is not", save="v7_attribution_verdict")
    note("ATTRIB", "Can competitor loss be separated from contraction?",
         "candidate two-axis split produced; unvalidated",
         "Counterparty survival is confounded by counterparty popularity, which is why the cut "
         "is the stayer median rather than an absolute. Validation needs CRM win/loss coding, "
         "and a proper measurement needs PAYS_CPTY.")


## 8 · The deliverable

In [ ]:
# =====================================================================
# 9 · QUEUE SPEC — self-contained HTML, no CDN
# =====================================================================
CSS = """
:root{--ink:#16181D;--mut:#6B7280;--line:#E5E7EB;--acc:#C1440E;--acc2:#4A6FA5;--bg:#FCFCFD}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);
     font:15px/1.65 "IBM Plex Sans",-apple-system,Segoe UI,sans-serif}
.wrap{max-width:1000px;margin:0 auto;padding:48px 28px 90px}
h1{font-size:30px;margin:0 0 6px;letter-spacing:-.4px}
h2{font-size:19px;margin:44px 0 6px;padding-top:20px;border-top:1px solid var(--line)}
.sub{color:var(--mut);font-size:13px;margin-bottom:34px}
.lede{font-size:16px;border-left:3px solid var(--acc);padding:2px 0 2px 16px;margin:18px 0 26px}
table.t{border-collapse:collapse;width:100%;font-size:13px;margin:12px 0}
table.t th{text-align:left;border-bottom:2px solid var(--ink);padding:7px 9px;font-weight:600}
table.t td{border-bottom:1px solid var(--line);padding:6px 9px}
table.t tr:hover td{background:#F6F7F9}
.warn{background:#FFF7ED;border-left:3px solid var(--acc);padding:12px 16px;
      font-size:13.5px;margin:16px 0}
.note{color:var(--mut);font-size:13px;margin:8px 0 0}
"""
_now = dt.datetime.now().strftime("%Y-%m-%d")
_q = QUEUE[QUEUE.k == QUEUE_K].sort_values("precision", ascending=False)
_qq = _q[["model", "k", "alerts_raw", "distinct_customers", "conversations", "tp",
          "precision", "conversations_per_tp", "median_lead_m"]].round(3)
_cap = (QUEUE[QUEUE.model == BEST_MODEL][["k", "alerts_raw", "tp", "precision",
                                          "conversations_per_tp", "median_lead_m"]].round(3))
_cov = (SHAPE[["m_idx", "at_risk", "cov_fin_out_n", "cov_bal_live"]].round(3))

body = f"""<div class='wrap'>
<h1>Deposit attrition — queue specification</h1>
<div class='sub'>PKG · PNC Treasury Management · Data Science · v7 · {_now} ·
 {len(ORIGINS)} rolling origins, m_idx {ORIGINS[0]}&ndash;{ORIGINS[-1]} ·
 horizon {PRIMARY_H} months · label {PRIMARY_DEF}</div>

<div class='lede'>Every number here is a <b>calendar-month</b> statistic: score the whole at-risk
book, rank inside the month, work the top K. Earlier runs reported event-time statistics, which
answer a different question and cannot be turned into a list.</div>

<div class='warn'><b>Queue coverage is the constraint nobody had priced.</b> A rule can only
alert on customers it can score. <code>fin_out_n</code> — the strongest single signal in the
study — has a <code>dd</code> for roughly a third of the book. The rest are invisible to it, and
no ordering table in v6 shows that.</div>

<h2>1 · What each rule delivers at {QUEUE_K} alerts a month</h2>
{tbl(_qq)}
<p class='note'>conversations collapses re-flags of the same customer inside
{COOLDOWN_M} months; conversations_per_tp is the cost figure to quote.</p>

<h2>2 · {BEST_MODEL} across capacities</h2>
{tbl(_cap)}

<h2>3 · Against the incumbent 30% rule, same frame</h2>
{tbl(pd.read_csv(OUT_DIR / 'v7_incumbent_queue.csv').round(3))}

<h2>4 · Two tiers, or one list taken deeper</h2>
{tbl(pd.read_csv(OUT_DIR / 'v7_two_tier.csv').round(4))}

<h2>5 · Scoreable share of the book, by month</h2>
{tbl(_cov)}

<h2>6 · What this does not establish</h2>
<ul>
<li>No counterparty data. Signal 12 is measured from the PNC-visible slice only; PAYS_CPTY and
CptyFinEntity would change it and every number here is designed to re-run unchanged when they land.</li>
<li>Competitor versus contraction is a candidate split with no ground truth. It needs CRM
win/loss coding before it is briefed.</li>
<li><code>segment_desc</code> is excluded throughout as a leakage risk and is not a
point-in-time attribute.</li>
</ul>
</div>"""
html = f"<!doctype html><html><head><meta charset='utf-8'><title>PKG attrition queue</title><style>{CSS}</style></head><body>{body}</body></html>"
(OUT_DIR / HTML_NAME).write_text(html, encoding="utf-8")
print(f"wrote {OUT_DIR / HTML_NAME}  ({len(html):,} bytes, self-contained)")
disp(pd.DataFrame(FINDINGS), title="8b &middot; v7 findings ledger", n=40)


---

## After this run

1. **§4a is the go/no-go on modelling.** Read AUC and `queue_coverage` together. If
   `M4_pay_plus_bal` wins on AUC *and* scores 100% of the book, §4.2 is closed and the next
   artefact is a scoring module, not another study. If `M2_fin_only` still wins on the rows where
   both are defined, the finding is that the model buys **coverage, not discrimination** — which
   is still the reason to ship it, and should be said in exactly those words.

2. **§5c decides whether the two-tier design survives.** The handoff proposes it; nobody had
   compared it against one ranked list taken to the same depth. If the single list wins, drop the
   second tier — it is complexity that costs a maintenance surface and buys nothing.

3. **§2 reclassifies whatever is still pinned at −18.** A feature that hits the window edge twice
   has no onset in a 31-month panel. It belongs with the standing marker in §11d of the results
   log — a watch-list criterion — and should stop being counted among the early-warning signals.

4. **§7 is not briefable.** Take a sample of the DISPLACEMENT quadrant to whoever holds win/loss
   coding and get 200 labels. Without them this is a plausible story and nothing more.

5. **Update the three handoff files** from `FINDINGS_v7.csv` and the CSVs in
   `attrition_v7/`. In particular `01 §2` still says the defensible lead is 2 months on dense
   features — that is an event-time claim, and §5's median lead is the calendar-time replacement
   for it. They are different numbers measuring different things and the brief should carry the
   second one.

6. **When `PAYS_CPTY` lands**, re-run §3–§6 unchanged. The risk set, the folds and the specs are
   all keyed on the feature list, so adding counterparty features is a change to `FEATS` and
   nothing else. That is the comparison the whole programme has been holding open.
